# Igniters-Tunde-Wey Week 6 — Exercise Solutions

Solved exercises from **Igniters_week6_exercises.ipynb**. Run from **week6** (or add week6 to path) so `pricer` and data paths work.

---
## Exercise 1 (Day 1) — Extra category or parse failures

**Task:** Load one additional category with ItemLoader and print count; or inspect rows where parse returned None.

**Solution (Option A):** Load one category (e.g. Electronics) with ItemLoader — slow if run fully. **Option B:** From the raw Appliances dataset, loop until you get a few `parse(..., "Appliances")` that are None and inspect those datapoints (e.g. missing price, price out of range, or too little text).

In [ ]:
# Option B: Find a few parse failures (run from week6 so pricer and dataset are available)
import sys
from pathlib import Path

week6 = Path("..").resolve() if (Path("..") / "pricer").exists() else Path(".").resolve()
sys.path.insert(0, str(week6))

from datasets import load_dataset
from pricer.parser import parse
from tqdm import tqdm

dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)
failures = []
for i in range(min(5000, len(dataset))):  # cap to avoid long run
    item = parse(dataset[i], "Appliances")
    if item is None:
        failures.append((i, dataset[i]))
        if len(failures) >= 3:
            break

print(f"Found {len(failures)} parse failures in first 5000 rows.")
for i, dp in failures:
    price = dp.get("price", "N/A")
    title = (dp.get("title") or "")[:60]
    print(f"  index {i}: price={price}, title={title}...")

---
## Exercise 2 (Day 2) — Change SYSTEM_PROMPT to 2 sentences

**Task:** Ask for "Description: 2 sentences" in the rewrite prompt and compare output.

**Solution:** In `../day2.ipynb` change the prompt to:
`Description: 2 sentences description` and `Details: 1 sentence on features`. Re-run the cell that sets SYSTEM_PROMPT and the completion cell for one item. The model will return a longer description; quality may be better or noisier depending on the product.

In [ ]:
# Snippet for day2 — alternative SYSTEM_PROMPT
SYSTEM_PROMPT_2_SENT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 2 sentences description
Details: 1 sentence on features"""

print("Use this in day2.ipynb; then run completion on items[0].full and compare length/quality.")

---
## Exercise 3 (Day 3) — Add feature and retrain LinearRegression

**Task:** Add one feature (e.g. title length) to get_features, retrain, and evaluate.

**Solution:** Add `"title_length": len(item.title)` to the dict in `get_features`. Append `'title_length'` to `feature_columns`. Re-run the training and `linear_regression_pricer` cells, then `evaluate(linear_regression_pricer, test)`. Result may improve slightly (more signal) or stay similar; if the new feature is noisy, metrics can worsen.

In [ ]:
# Snippet: extended get_features for day3
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight == 0 else 0,
        "text_length": len(item.summary or ""),
        "title_length": len(item.title),  # new feature
    }

# In the training cell, use: feature_columns = ['weight', 'weight_unknown', 'text_length', 'title_length']
print("Use in day3.ipynb; retrain LinearRegression and run evaluate(linear_regression_pricer, test).")

---
## Exercise 4 (Day 4) — Compare neural net vs LLM pricer on a subset

**Task:** On test[:20], compare average absolute error of neural net pricer vs LLM pricer.

**Solution:** After training the neural net and defining the LLM pricer in day4, run the loop below. Replace `neural_net_pricer` and `llm_pricer` with your actual function names from the notebook.

In [ ]:
# Run from day4 notebook after defining both pricers and loading test.
# Substitute your pricer names.

subset = test[:20]

def mean_abs_error(pricer, data):
    errors = [abs(pricer(item) - item.price) for item in data]
    return sum(errors) / len(errors) if errors else 0

# If you have neural_net_pricer and llm_pricer in the notebook:
# nn_err = mean_abs_error(neural_net_pricer, subset)
# llm_err = mean_abs_error(llm_pricer, subset)
# print(f"Neural net mean abs error (n=20): ${nn_err:.2f}")
# print(f"LLM pricer mean abs error (n=20): ${llm_err:.2f}")
# print("Better:", "Neural net" if nn_err < llm_err else "LLM")

print("Uncomment and use your pricer names in day4; then run the cell.")

---
## Exercise 5 (Day 5) — Compare fine-tuned vs Day 4 LLM

**Task:** Run evaluate on the fine-tuned pricer and compare to Day 4 zero-shot LLM.

**Solution:** After the fine-tune job completes, define a pricer that calls the fine-tuned model (e.g. `ft:gpt-4.1-nano:...`) and run `evaluate(fine_tuned_pricer, test, size=200)`. Compare the reported cumulative average error (or summary) with the Day 4 LLM run. Typically the fine-tuned model does better because it was trained on the same style of product/price pairs; the improvement depends on data size and quality.

In [ ]:
# Example: after you have the fine-tuned model name (e.g. from openai.fine_tuning.jobs.retrieve(job_id))
# def fine_tuned_pricer(item):
#     response = openai.chat.completions.create(model=FINETUNED_MODEL_ID, messages=[{"role": "user", "content": f"Estimate the price...\n\n{item.summary}"}])
#     text = response.choices[0].message.content
#     return float(re.search(r'[\d.]+', text.replace(",", "")).group())
# evaluate(fine_tuned_pricer, test, size=200)

print("Use your fine-tuned model id in day5; run evaluate and compare the output to Day 4 LLM evaluator run.")

---
## Summary

| Exercise | Takeaway |
|----------|----------|
| 1 | Parse fails on missing/out-of-range price or too little text; ItemLoader loads by category. |
| 2 | Changing SYSTEM_PROMPT changes rewrite length/format; 2 sentences can improve or add noise. |
| 3 | Adding a feature (e.g. title_length) and retraining is straightforward; measure with evaluate(). |
| 4 | Mean absolute error on a small subset compares two pricers without full evaluate(). |
| 5 | Fine-tuned model usually beats zero-shot LLM on the same test set; run evaluate() for both. |